In [ ]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess

# Define mounted checkpoint locations (Update these paths to match your Kaggle Inputs)
MOUNTED_SATEHAZE_WEIGHTS = '/kaggle/input/YOUR_FAILED_RUN_SLUG/weights/SateHaze1k_Baseline/model_best.pth'
MOUNTED_RRSHID_WEIGHTS = '/kaggle/input/YOUR_NEW_RUN_SLUG/weights/RRSHID_Baseline/model_best.pth'

# 1. Clone patched repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# 2. Setup RRSHID dataset
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}

# 3. Virtual Dataset Aggregator (Test Splits Only)
UNIFIED_SATEHAZE = '/kaggle/working/Unified_SateHaze1k'
UNIFIED_RRSHID = '/kaggle/working/Unified_RRSHID'

def prepare_unified_layout(source_base, unified_base, levels, splits):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    for split in splits:
        hazy_dir = os.path.join(unified_base, split, 'hazy')
        gt_dir = os.path.join(unified_base, split, 'GT')
        os.makedirs(hazy_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        
        for level in levels:
            raw_split = os.path.join(source_base, level, split)
            if not os.path.isdir(raw_split): continue
            raw_hazy = os.path.join(raw_split, 'hazy')
            raw_gt = next((os.path.join(raw_split, c) for c in ['GT', 'gt', 'clear'] if os.path.isdir(os.path.join(raw_split, c))), None)
            
            if not raw_gt or not os.path.isdir(raw_hazy): continue
            
            for fname in os.listdir(raw_hazy):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
                unique_fname = f"{level}_{fname}"
                os.symlink(os.path.join(raw_hazy, fname), os.path.join(hazy_dir, unique_fname))
                os.symlink(os.path.join(raw_gt, fname), os.path.join(gt_dir, unique_fname))

prepare_unified_layout('/kaggle/input/datasets/xuxingxing233/satehaze1k', UNIFIED_SATEHAZE, 
                       ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick'], ['test'])
prepare_unified_layout(RRSHID_BASE, UNIFIED_RRSHID, 
                       ['thin_fog', 'moderate_fog', 'thick_fog'], ['test'])

# 4. Test Orchestrator
def run_test(dataset_name, unified_base, weights_path):
    print(f"\n{'='*50}\nTesting Baseline: {dataset_name}\n{'='*50}")
    result_dir = f'/kaggle/working/results/{dataset_name}_Baseline'
    os.makedirs(result_dir, exist_ok=True)
    
    test_cmd = [
        'python', 'infer.py',
        '--test_input', f'{unified_base}/test/hazy',
        '--test_target', f'{unified_base}/test/GT',
        '--weights', weights_path,
        '--result_path', result_dir
    ]
    print("-> Running Inference...")
    subprocess.run(test_cmd, check=True)
    
    eval_cmd = [
        'python', 'evaluate.py',
        '--train_folder', result_dir,
        '--target_folder', f'{unified_base}/test/GT'
    ]
    print("-> Computing Evaluation Metrics...")
    subprocess.run(eval_cmd, check=True)

# Execute Testing Pipeline
run_test("SateHaze1k", UNIFIED_SATEHAZE, MOUNTED_SATEHAZE_WEIGHTS)
run_test("RRSHID", UNIFIED_RRSHID, MOUNTED_RRSHID_WEIGHTS)

print("\nTesting and Evaluation Complete! Results available in /kaggle/working/results/")